### The DeepRitz-Method

While PINNs usually utilize the strong formulation of the PDE, there are other concepts that use different formulations of the PDE for training the network, one of them is the DeepRitz method. There, the energy formulation
of the problem is used.

Let us consider the following simple Poisson equation:
\begin{align*}
    -\Delta u &= -1 \text{ in } \Omega, \\
    u &= 0 \text{ on } \partial \Omega,
\end{align*} 

with $\Omega=[0, 1] \times [0, 1]$.

The energy functional for this problem is given by:
\begin{equation}
    E(v) = \int_\Omega \frac{1}{2}\|\nabla v(x)\|^2 \text{d}x  + \int_\Omega v(x) \text{d}x
\end{equation}
The solution $u$ of the strong/weak formulation minimizes the energy functional, namely $E(u) \leq E(v)$ for all $v$ in the corresponding function space. The DeepRitz method tries to find the minimizer of the energy functional $E$.

In  practice, just minimizing the energy functional above, does not automatically enforce the boundary condition. For that, one can minimize
\begin{equation}
\tilde{E}(v) := w_1 E(v) + w_2\int_{\partial \Omega} v^2(x) \text{d}x,
\end{equation}
where $w_1,w_2$ are some weights.

In [1]:
# This block is for GPU selection. Please execute.
import pathlib
import os
user = int(str(pathlib.Path().resolve())[24:26])
os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [2]:
import torchphysics as tp 
import torch
import pytorch_lightning as pl

# Create input and outout spaces
X = tp.spaces.R2('x') 
U = tp.spaces.R1('u')

In [ ]:
# TODO: Implement the square [0, 1] x [0, 1] 
square = ...

In [ ]:
# Here we create the sampler. For DeepRitz we need to approximate the integral, therefore one usually needs more points
# then in the PINN approach to get a good estimate.
# But since in the energy functional the derivatives are of lower order, this generally does not lead to memory problems.
inner_sampler = tp.samplers.RandomUniformSampler(square, n_points=100000) 

# TODO: Create a sampler for the boundary, which samples 40000 points at each iteration.
bound_sampler = ...

The classic DeepRitz method uses a ResNet-architecture instead of a FCN. This is also implemented and used here: 

In [ ]:
# TODO: Add the input and output space
model = tp.models.DeepRitzNet(..., width=20, depth=4)

The transformation of the mathematical equations is handled similar to the PINN approach. For each integral in the energy functional, we define a condition.

Instead of a *PINNCondition*, we now use a *DeepRitzCondition*. While the *PINNCondition* computes the mean of the squared outputs of the residual function, the *DeepRitzCondition* only averages the output of the residual function (without computing squares, which allows negative losses).

In [ ]:
# The integrand function for the boundary integral
def bound_residual(u):
    return u**2

bound_cond = tp.conditions.DeepRitzCondition(module=model, sampler=bound_sampler, 
                                             integrand_fn=bound_residual, weight=100)

In [ ]:
# The integrand function for the integral over Omega.
# TODO: Following the implementation in the above cell, complete the condition below:
#       Hint: to compute the norm over a batch of vectors called v, the call
#       torch.sum(v, dim=1, keepdim=True) is useful.
def energy_residual(u, x):
    return ...

pde_cond = tp.conditions.DeepRitzCondition(..., weight=1.0)

In [ ]:
learning_rate = 1e-3
training_iterations = 2000

optim = tp.OptimizerSetting(optimizer_class=torch.optim.Adam, lr=learning_rate)
solver = tp.solver.Solver(train_conditions=[bound_cond, pde_cond], optimizer_setting=optim)

trainer = pl.Trainer(devices=1, accelerator="gpu",
                     max_steps=training_iterations,
                     logger=False,
                     benchmark=True,
                     enable_checkpointing=False)

trainer.fit(solver)

In [ ]:
plot_sampler = tp.samplers.PlotSampler(plot_domain=square, n_points=640, device='cuda')
fig = tp.utils.plot(model, lambda u : u, plot_sampler, plot_type='contour_surface')